# Fine-Tune Cellpose
Fine-tunes Cellpose cyto3 on hand-annotated patches, evaluates against a held-out test set.

**Upload 4 zips:** `train_patches.zip`, `train_masks.zip`, `test_patches.zip`, `test_masks.zip`  
**Output:** Fine-tuned model + AP@0.5 comparison + predictions for model-assisted annotation

In [ ]:
!pip install -q "opencv-python-headless>=4.9.0.80" cellpose==3.1.1.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, glob, time
from cellpose import core, models, train, metrics
from skimage import measure

use_GPU = core.use_gpu()
print(f"GPU: {['NO','YES'][use_GPU]}")

## Upload Data
Each zip should contain `.npy` files at the top level.  
Indices must match: `img_0000.npy` ↔ `mask_0000.npy`

In [ ]:
from google.colab import files
uploaded = files.upload()

!mkdir -p /content/data/{train_imgs,train_masks,test_imgs,test_masks}
!unzip -qoj train_patches.zip -d /content/data/train_imgs
!unzip -qoj train_masks.zip   -d /content/data/train_masks
!unzip -qoj test_patches.zip  -d /content/data/test_imgs
!unzip -qoj test_masks.zip    -d /content/data/test_masks

print(f"Train images: {len(glob.glob('/content/data/train_imgs/img_*.npy'))}")
print(f"Train masks:  {len(glob.glob('/content/data/train_masks/mask_*.npy'))}")
print(f"Test images:  {len(glob.glob('/content/data/test_imgs/img_*.npy'))}")
print(f"Test masks:   {len(glob.glob('/content/data/test_masks/mask_*.npy'))}")

In [ ]:
def load_paired(img_dir, mask_dir):
    """Load image-mask pairs, matching by index."""
    mask_files = sorted(glob.glob(os.path.join(mask_dir, 'mask_*.npy')))
    images, masks = [], []
    for mf in mask_files:
        idx = os.path.basename(mf).replace('mask_', '').replace('.npy', '')
        img_path = os.path.join(img_dir, f'img_{idx}.npy')
        assert os.path.exists(img_path), f"Missing image for {mf}"
        images.append(np.load(img_path))
        masks.append(np.load(mf))
    return images, masks

train_images, train_masks = load_paired('/content/data/train_imgs', '/content/data/train_masks')
test_images, test_masks   = load_paired('/content/data/test_imgs',  '/content/data/test_masks')

print(f"Train: {len(train_images)} patches")
print(f"Test:  {len(test_images)} patches")

## Measure Cell Diameter
Computed from hand labels. Do NOT use `diameter=None` — Cellpose guesses wrong on phase contrast.

In [ ]:
all_diameters = []
for mask in train_masks + test_masks:
    for p in measure.regionprops(mask):
        all_diameters.append(np.sqrt(4 * p.area / np.pi))

all_diameters = np.array(all_diameters)
DIAMETER = int(np.round(np.median(all_diameters)))

print(f"Cells measured: {len(all_diameters)}")
print(f"Median diameter: {np.median(all_diameters):.1f} px → DIAMETER = {DIAMETER}")

plt.hist(all_diameters, bins=40, color='steelblue', edgecolor='black')
plt.axvline(DIAMETER, color='red', linestyle='--', label=f'median={DIAMETER}')
plt.xlabel('equivalent diameter (px)')
plt.ylabel('count')
plt.legend()
plt.show()

## Baseline

In [ ]:
base_model = models.CellposeModel(gpu=use_GPU, model_type='cyto3')
base_preds, _, _ = base_model.eval(test_images, diameter=DIAMETER, channels=[0, 0])

base_ap = []
for pred, gt in zip(base_preds, test_masks):
    ap, _, _, _ = metrics.average_precision([gt], [pred], threshold=[0.5])
    base_ap.append(ap[0][0])

BASE_AP = np.mean(base_ap)
print(f"Base cyto3 AP@0.5: {BASE_AP:.3f}  (range: {np.min(base_ap):.3f} – {np.max(base_ap):.3f})")

## Fine-Tune
Defaults found via hyperparameter sweep on phase contrast data. Adjust if needed.

In [ ]:
N_EPOCHS       = 500
LEARNING_RATE  = 0.001
WEIGHT_DECAY   = 1e-5
NIMG_PER_EPOCH = len(train_images)

os.makedirs('/content/models', exist_ok=True)
model = models.CellposeModel(gpu=use_GPU, model_type='cyto3')

t0 = time.time()
model_path = train.train_seg(
    model.net,
    train_data     = train_images,
    train_labels   = train_masks,
    channels       = [0, 0],
    normalize      = True,
    save_path      = '/content/models',
    n_epochs       = N_EPOCHS,
    learning_rate  = LEARNING_RATE,
    weight_decay   = WEIGHT_DECAY,
    nimg_per_epoch = NIMG_PER_EPOCH,
    model_name     = 'finetuned_cyto3'
)
print(f"Done in {(time.time() - t0) / 60:.1f} min — saved to {model_path[0]}")

## Evaluate

In [ ]:
finetuned = models.CellposeModel(gpu=use_GPU, pretrained_model=str(model_path[0]))
ft_preds, _, _ = finetuned.eval(test_images, diameter=DIAMETER, channels=[0, 0])

ft_ap = []
for pred, gt in zip(ft_preds, test_masks):
    ap, _, _, _ = metrics.average_precision([gt], [pred], threshold=[0.5])
    ft_ap.append(ap[0][0])

FT_AP = np.mean(ft_ap)
print(f"Base cyto3       AP@0.5: {BASE_AP:.3f}")
print(f"Fine-tuned       AP@0.5: {FT_AP:.3f}")
print(f"Delta:                   {FT_AP - BASE_AP:+.3f}")

In [ ]:
# Per-patch breakdown
print(f"{'Patch':<7} {'Base':<9} {'Tuned':<9} {'Delta':<9} {'GT':<6} {'Pred':<6}")
print('-' * 46)
for i in range(len(test_images)):
    n_gt = len(np.unique(test_masks[i])) - 1
    n_ft = len(np.unique(ft_preds[i])) - 1
    d    = ft_ap[i] - base_ap[i]
    print(f"{i:<7} {base_ap[i]:<9.3f} {ft_ap[i]:<9.3f} {d:<+9.3f} {n_gt:<6} {n_ft:<6}")

In [ ]:
# Visual comparison
n   = min(6, len(test_images))
idx = np.random.choice(len(test_images), n, replace=False)
fig, axes = plt.subplots(4, n, figsize=(3 * n, 12))

for col, i in enumerate(idx):
    for row, (data, label) in enumerate([
        (test_images[i], 'image'),
        (test_masks[i] > 0, f'hand ({len(np.unique(test_masks[i]))-1})'),
        (base_preds[i] > 0, f'base AP={base_ap[i]:.2f}'),
        (ft_preds[i] > 0,   f'tuned AP={ft_ap[i]:.2f}'),
    ]):
        axes[row][col].imshow(data, cmap='gray')
        axes[row][col].set_title(label, fontsize=9)
        axes[row][col].axis('off')

plt.suptitle(f'base ({BASE_AP:.3f}) vs fine-tuned ({FT_AP:.3f})', fontsize=11)
plt.tight_layout()
plt.show()

## Export Model + Predictions

In [ ]:
import shutil

# Copy model
shutil.copy2(str(model_path[0]), '/content/finetuned_best')

# Save training summary
with open('/content/training_summary.txt', 'w') as f:
    f.write(f"Date:          {time.strftime('%Y-%m-%d %H:%M')}\n")
    f.write(f"Train patches: {len(train_images)}\n")
    f.write(f"Test patches:  {len(test_images)}\n")
    f.write(f"Diameter:      {DIAMETER}\n")
    f.write(f"Config:        ep={N_EPOCHS} lr={LEARNING_RATE} wd={WEIGHT_DECAY}\n")
    f.write(f"Base AP@0.5:   {BASE_AP:.3f}\n")
    f.write(f"Tuned AP@0.5:  {FT_AP:.3f} ({FT_AP - BASE_AP:+.3f})\n")

!zip -qj /content/finetuned_best.zip /content/finetuned_best /content/training_summary.txt

from google.colab import files
files.download('/content/finetuned_best.zip')
print("Model exported. Place in models/ directory of the repo.")

### Generate Predictions for Annotation
Run the fine-tuned model on ALL patches (not just test) and export predictions.  
Upload a zip of patches → get back `pred_masks.zip` for `annotate.py -p`.

In [ ]:
# Upload patches to generate predictions for
print("Upload a zip of img_*.npy patches to generate predictions for:")
pred_upload = files.upload()
pred_zip = list(pred_upload.keys())[0]

!mkdir -p /content/pred_input
!unzip -qoj "{pred_zip}" -d /content/pred_input

pred_files = sorted(glob.glob('/content/pred_input/**/img_*.npy', recursive=True))
print(f"Found {len(pred_files)} patches")

In [ ]:
pred_imgs = [np.load(f) for f in pred_files]
pred_masks, _, _ = finetuned.eval(pred_imgs, diameter=DIAMETER, channels=[0, 0])

os.makedirs('/content/pred_out', exist_ok=True)
for f, mask in zip(pred_files, pred_masks):
    name = os.path.basename(f).replace('img_', 'pred_')
    np.save(f'/content/pred_out/{name}', mask.astype(np.int32))

!zip -qj /content/pred_masks.zip /content/pred_out/*.npy
files.download('/content/pred_masks.zip')
print(f"Exported {len(pred_masks)} predictions. Unzip into pred_masks/ and use with annotate.py -p")